In [1]:
# Have to use my_gbd_sunset environment
import os
# Shared Functions
from db_queries import get_location_metadata, get_age_metadata, get_population, get_cause_metadata, get_outputs, get_covariate_estimates
from get_draws.api import get_draws
import pandas as pd

In [ ]:
import sys
from pathlib import Path
from datetime import date
import json

sys.path.insert(0, '/ihme/homes/bcreiner/repos/idd-forecast-mbp/src')
from idd_forecast_mbp import constants as mbpc

data_date = '2026_04_02'
data_path = mbpc.GBD_DATA_PATH / data_date
data_path.mkdir(parents=True, exist_ok=True)
with open(data_path / "gbd_constants.json", "w") as f:
    json.dump(mbpc.gbd_constants, f, indent=2)

make_current = False
if make_current:
    current_link = mbpc.GBD_DATA_PATH / "current"
    if current_link.is_symlink() or current_link.exists():
        current_link.unlink()
    current_link.symlink_to(data_date)

In [8]:
gbd_location_set_id = 35
fhs_location_set_id = 39
fhs_hierarchy_2023 = get_location_metadata(location_set_id = fhs_location_set_id, release_id = mbpc.gbd_constants['release_2023_id'])
gbd_hierarchy_2023 = get_location_metadata(location_set_id = gbd_location_set_id, release_id = mbpc.gbd_constants['release_2023_id'])
cols_to_drop = ["start_date", "end_date", "date_inserted", "last_updated", "last_updated_by", "last_updated_action"]
fhs_hierarchy_2023 = fhs_hierarchy_2023.drop(columns=cols_to_drop)
gbd_hierarchy_2023 = gbd_hierarchy_2023.drop(columns=cols_to_drop)

location_ids = gbd_hierarchy_2023[gbd_hierarchy_2023['level'] <= 3]['location_id'].tolist()

years = list(range(2000, 2024))
sex_ids = [1, 2, 3]


In [9]:
age_metadata = get_age_metadata(release_id = mbpc.gbd_constants['release_2023_id'])
age_group_ids = [1, 22, 27] + age_metadata['age_group_id'].tolist()
# Get population
gbd_population = get_population(
    age_group_id=age_group_ids,
    release_id=mbpc.gbd_constants['release_2023_id'],
    year_id=years,
    location_id=location_ids,
    sex_id=sex_ids
)
gbd_population = gbd_population[["age_group_id", "location_id", "year_id", "sex_id", "population"]].copy()

In [ ]:
for cause_key, cause_info in mbpc.cause_map.items():
    print(f"Processing cause: {cause_info['cause_name']}")
    cause_id = cause_info['cause_id']
    cause_name = cause_info['cause_name']
    fhs_cause_name = cause_info['fhs_cause_name']

    print(f"Getting AA results for cause: {cause_name}")
    aa_results = get_outputs(
        "cause",
        cause_id=cause_id,
        measure_id=[1,2,3,4,5,6],
        year_id=years,
        location_id=location_ids,
        age_group_id=[22],
        release_id=mbpc.gbd_constants['release_2023_id'],
        metric_id=[1, 3],
        sex_id=[1, 2, 3],
        compare_version_id=mbpc.gbd_constants['compare_2023_v']
    )
    aa_results = aa_results.merge(gbd_hierarchy_2023, on="location_id", how="left")
    aa_results = aa_results.merge(gbd_population, on=["age_group_id", "location_id", "year_id", "sex_id"], how="left")
    aa_results.to_parquet(f"{data_path}/aa_{cause_key}_results.parquet", index=False)

    print(f"Getting AS results for cause: {cause_name}")
    as_results = get_outputs(
        "cause",
        cause_id=cause_id,
        measure_id=[1,6],
        year_id=years,
        location_id=location_ids,
        age_group_id=age_group_ids,
        release_id=mbpc.gbd_constants['release_2023_id'],
        metric_id=[1, 3],
        sex_id=[1, 2, 3],
        compare_version_id=mbpc.gbd_constants['compare_2023_v']
    )

    as_results = as_results.merge(gbd_hierarchy_2023, on="location_id", how="left")
    as_results = as_results.merge(gbd_population, on=["age_group_id", "location_id", "year_id", "sex_id"], how="left")
    as_results.to_parquet(f"{data_path}/as_{cause_key}_results.parquet", index=False)

Processing cause: Malaria
Getting AA results for cause: Malaria
Getting AS results for cause: Malaria
Processing cause: Malaria falciparum
Getting AA results for cause: Malaria falciparum


RuntimeError: no results found. Tables searched were ['output_epi_single_year_v20291_incidence', 'output_cod_single_year_v20467', 'output_epi_single_year_v20291_yld', 'output_summary_single_year_v20476_daly', 'output_epi_single_year_v20291_prevalence']. Filters used were {'topic': 'cause', 'compare_version_id': [8352], 'release_id': [16], 'process_version_id': [20467, 20321, 20373, 20320, 20484, 20488, 20476, 20487, 20291, 20493, 20456, 20474, 20482, 20324, 20465, 20486, 20478, 20483, 20344, 20475, 20472, 20473, 20477], 'version': 'best', 'conn_def': 'modeling-gbd-db-read', 'year_id': [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023], 'location_set_id': [35], 'location_id': [1, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 50, 49, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 70, 71, 72, 65, 66, 67, 68, 69, 100, 101, 349, 102, 96, 97, 98, 99, 73, 74, 75, 76

In [13]:
temp_aa_results = get_outputs(
        "cause",
        cause_id=mbpc.cause_map['malaria_pf']['cause_id'],
        measure_id=[1,6],
        year_id=years,
        location_id=location_ids,
        age_group_id=[22],
        release_id=mbpc.gbd_constants['release_2023_id'],
        metric_id=[1, 3],
        sex_id=[1, 2, 3],
        compare_version_id=mbpc.gbd_constants['compare_2023_v']
    )

RuntimeError: no results found. Tables searched were ['output_epi_single_year_v20291_incidence', 'output_cod_single_year_v20467']. Filters used were {'topic': 'cause', 'compare_version_id': [8352], 'release_id': [16], 'process_version_id': [20467, 20321, 20373, 20320, 20484, 20488, 20476, 20487, 20291, 20493, 20456, 20474, 20482, 20324, 20465, 20486, 20478, 20483, 20344, 20475, 20472, 20473, 20477], 'version': 'best', 'conn_def': 'modeling-gbd-db-read', 'year_id': [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023], 'location_set_id': [35], 'location_id': [1, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 50, 49, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 70, 71, 72, 65, 66, 67, 68, 69, 100, 101, 349, 102, 96, 97, 98, 99, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 367, 89, 90, 91, 396, 92, 93, 94, 95, 103, 120, 121, 122, 123, 104, 105, 